### Búsqueda de documentos por índice inverso

#### Búsqueda de documentos por palabras

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import  Counter

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
import nltk
nltk.download(['punkt','averaged_perceptron_tagger','wordnet'])

from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize, pos_tag
from nltk.corpus import wordnet
from nltk.corpus.reader.wordnet import NOUN, VERB, ADV, ADJ

Cargamos los datos

In [ ]:
db = fetch_20newsgroups(remove=('headers','footers','quotes'))

In [ ]:
len(db.data)

In [ ]:
db.data[0]

In [ ]:
morphy_tag = {
    'JJ' : ADJ,
    'JJR' : ADJ,
    'JJS' : ADJ,
    'VB' : VERB,
    'VBD' : VERB,
    'VBG' : VERB,
    'VBN' : VERB,
    'VBP' : VERB,
    'VBZ' : VERB,
    'RB' : ADV,
    'RBR' : ADV,
    'RBS' : ADV
}

def doc_a_tokens(doc):
    tagged = pos_tag(word_tokenize(doc.lower()))
    lemmatizer = WordNetLemmatizer()
    tokens = []
    for p,t in tagged:
        tokens.append(lemmatizer.lemmatize(p, pos=morphy_tag.get(t, NOUN)))

    return tokens

Guardamos el conjunto preprocesado como una lista de cadenas, una por documento

In [ ]:
corpus = []
for d in db.data[:100]:
    d = d.replace('\n',' ').replace('\r',' ').replace('\t',' ')
    tokens = doc_a_tokens(d)
    corpus.append(' '.join(tokens))

In [ ]:
corpus[0]

Obtenemos las bolsas de palabras de los documentos preprocesados usando la clase CountVectorizer de scikit-learn

In [ ]:
v = CountVectorizer(stop_words='english', max_features=5000, max_df=0.8)
bolsas = v.fit_transform(corpus)

In [ ]:
print('Componentes de primer documento: {0}'.format(corpus[0]))

In [ ]:
print('Bolsa de primer documento: [\n{0}]'.format(bolsas[0]))

Definimos la clase para el índice inverso con un método para recuperar los documentos que contienen una lista de palabras

In [ ]:
class IndiceInverso:
    def  __getitem__(self, idx):
        return self.ifs[idx]

    def __repr__(self):
        contenido = ['%d::%s' % (i, self.ifs[i]) for i in range(len(self.ifs))]
        return "<IFS :%s >" % ('\n'.join(contenido))

    def __str__(self):
        contenido = ['%d::%s' % (i, self.ifs[i]) for i in range(len(self.ifs))]
        return '\n'.join(contenido)

    def recupera(self, l):
        return Counter([j for (i,_) in l for j in self.ifs[i]])

    def from_csr(self, csr):
        self.ifs = [[] for _ in range(csr.shape[1])]
        coo = csr.tocoo()    
        for i,j,v in zip(coo.row, coo.col, coo.data):
            self.ifs[j].append(i)

Instanciamos nuestra clase IndiceInverso y creamos la estructura a partir de nuestras bolsas de palabras

In [ ]:
ifs = IndiceInverso()
ifs.from_csr(bolsas)

Definimos una función que convierta de arreglos dispersos CSR a listas de listas

In [ ]:
def csr_to_ldb(csr):
    ldb = [[] for _ in range(csr.shape[0])]
    coo = csr.tocoo()    
    for i,j,v in zip(coo.row, coo.col, coo.data):
        ldb[i].append((j, v))

    return ldb



Generamos algunas consultas y calculamos sus bolsas de palabras



In [ ]:
consultas = []
for c in ['nasa space mission satellite','government crime enforcement security']:
    tokens = doc_a_tokens(c)
    consultas.append(' '.join(tokens))

In [ ]:
consultas

In [ ]:
bc = v.transform(consultas)
cl = csr_to_ldb(bc)

Usamos el índice inverso para recuperar los documentos que contienen las palabras de la primera consulta ordenados por coincidencias y visualizamos el primer documento recuperado

In [ ]:
recs = ifs.recupera(cl[0])
top = recs.most_common()[0]
recs.most_common()

In [ ]:
db.data[top[0]]



Repetimos el proceso anterior para la segunda consulta


In [ ]:
recs = ifs.recupera(cl[1])
top = recs.most_common()[0]
print(recs.most_common())
print(db.data[top[0]])

#### Búsqueda de documentos similares

Ahora vamos a realizar búsquedas de documentos similares a un documento de consulta.

Primero tomamos 1 documento que sirva de consulta y lo visualizamos

In [ ]:
dc = db.data[0]
dc

Obtenemos su bolsa

In [ ]:
tokens = doc_a_tokens(dc)
bolsa_dc = v.transform([' '.join(tokens)])
print('Componentes para consulta: {0}'.format(tokens))
print('Bolsa para consulta: [\n{0}]'.format(bolsa_dc))

Definimos una función para hacer búsqueda por fuerza bruta dada una función de distancia o similitud

In [ ]:
def fuerza_bruta(base, consulta, fd):
    medidas = np.zeros(base.shape[0])
    for i,x in enumerate(base):
        medidas[i] = fd(consulta, x)

    return medidas

Definimos la función para la similitud coseno

In [ ]:
def similitud_coseno(x, y):
    x = x.toarray()[0]
    y = y.toarray()[0]
    pnorma = (np.sqrt(x @ x) * np.sqrt(y @ y))

    if pnorma > 0:
        return (x @ y) / pnorma
    else: 
        return np.nan 

In [ ]:
sims = fuerza_bruta(bolsas[1:], bolsa_dc, similitud_coseno)

In [ ]:
print('Similitud máxima es {0} de documento {1}'.format(np.nanmax(sims), np.nanargmax(sims)+ 1))



Revisamos documento más similar



In [ ]:
print(db.data[np.nanargmax(sims) + 1])

Definimos la distancia euclidiana

In [ ]:
def distancia_euclidiana(x, y):   
    x = x.toarray()[0]
    y = y.toarray()[0]
    return np.sqrt(np.sum((x - y)**2))

Repetimos el proceso anterior con la distancia euclidiana

In [ ]:
euclids = fuerza_bruta(bolsas[1:], bolsa_dc, distancia_euclidiana)
print('Distancia mínima es {0} de documento {1}'.format(np.nanmin(euclids), np.nanargmin(euclids) + 1))

Visualizamos el documento

In [ ]:
print(db.data[np.nanargmin(euclids) + 1])